In [1]:
!pip -q install -U pandas numpy scikit-learn tqdm transformers torch openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.1 which is incompatible.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.4.3 which is incompatible.


In [2]:
!pip install -U \
numpy==2.2.0 \
transformers==4.44.2 \
openai==1.99.9 \
scikit-learn \
tqdm \
torch

  Using cached numpy-2.2.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached openai-1.99.9-py3-none-any.whl.metadata (29 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.19.1.tar.gz (321 kB)
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Installing backend dependencies ... one
  Preparing metadata (pyproject.todone
Using cached numpy-2.2.0-cp313-cp313-macosx_14_0_arm64.whl (5.1 MB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
Using cached openai-1.99.9-py3-none-any.whl (786 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
error... -
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [91 lines of output]
      Running `maturin pep517 build-wheel -i /opt/anaconda3/bin/python3.13 --compatibi

In [3]:
import numpy
import transformers
import openai
import torch
import sklearn
import pandas as pd

print("Environment is clean and working.")

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Environment is clean and working.


In [4]:
import os

current_dir = os.getcwd()
print("Current working directory:", current_dir)

Current working directory: /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment


In [4]:
DATA_PATH = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/stress_test_eval_set.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(8000, 10)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,word_count,is_clean
0,ab7fd34bb8af1251,I dont know who you fucking think you are you ...,1,1,1,0,1,0,53,0
1,5b7bd7b09b7ba86c,IDIOT/////IDIOT/////IDIOT/////IDIOT/////IDIOT/...,1,1,1,0,1,0,5,0
2,588dc710fe713a8d,"Motherfucked\n\nyou PORKI muslim, what changes...",1,0,1,0,1,1,38,0
3,968c9e95dbee0e3d,: Im sorry didn't they talk shit to me first? ...,1,0,1,0,0,0,38,0
4,bf846d64e7ad8941,I posted my question here: Wikipedia:Help desk...,0,0,0,0,0,0,21,1


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/stress_test_eval_set.csv')
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Strategy: pick best examples per label combination
# For rare labels, take ALL examples. For common ones, sample.
pool = []

# 1. Always include ALL threat examples (only 34 — too rare to subsample)
pool.append(df[df['threat'] == 1])

# 2. All severe_toxic (96 examples)
pool.append(df[df['severe_toxic'] == 1])

# 3. All identity_hate (86 examples)  
pool.append(df[df['identity_hate'] == 1])

# 4. Sample from common combos to keep pool manageable
common = df[df[label_cols].sum(axis=1) == 0].sample(200, random_state=42)  # clean
pool.append(common)
pool.append(df[df[label_cols].sum(axis=1) >= 2].sample(100, random_state=42))  # multi-label

example_pool = pd.concat(pool).drop_duplicates(subset='id').reset_index(drop=True)
print(f"Pool size: {len(example_pool)}")
print(f"Label distribution in pool:\n{example_pool[label_cols].sum()}")

Pool size: 3147
Label distribution in pool:
toxic            2820
severe_toxic     1563
obscene          2368
threat            458
insult           2347
identity_hate    1381
dtype: int64


# zero shot

In [3]:
pip install openai scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install aiohttp

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [5]:
import nest_asyncio
nest_asyncio.apply()   # patches the running Jupyter event loop

# USE 8000 DATA STRESS EVAL TEST SET

In [ ]:
"""
Toxicity Multi-Label Classifier using GPT-4.1 API
Business Analytics Tool | Zero-Shot Prompt
Features: Async Parallel Calls (20x faster) + Checkpoint/Resume (crash-safe)

FIXES APPLIED:
  1. Comment sanitization — strips newlines, escapes curly braces before .format()
  2. Robust JSON extraction — regex pulls JSON even if GPT adds extra text
  3. max_tokens increased 100 → 150 to prevent truncated JSON responses
  4. System message added to reinforce JSON-only output from the model
  5. Retry logic now re-raises cleanly and logs the raw response on parse failure
"""

import os
import re
import json
import time
import asyncio
import aiohttp
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from typing import Optional

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = "gpt-4.1"
BATCH_SIZE = 500          # Checkpoint saved every 500 rows
MAX_CONCURRENT = 20       # Parallel async API calls
MAX_RETRIES = 3
RETRY_DELAY = 2           # seconds (doubles on each retry)
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Paths — update BASE_DIR to your local folder
BASE_DIR = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment"
INPUT_PATH  = f"{BASE_DIR}/stress_test_eval_set.csv"
OUTPUT_PATH = f"{BASE_DIR}/predictions_output3.csv"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"

# ── FIX 4: system message reinforces JSON-only output ──────────────────────────
SYSTEM_MESSAGE = (
    "You are a JSON-only classifier. "
    "Respond ONLY with a valid JSON object. "
    "No explanation, no markdown fences, no extra text before or after the JSON."
)

ZERO_SHOT_PROMPT = """SYSTEM ROLE: You are a strict multi-label text classifier specialising in toxic content detection for online comments.

TASK: Classify the given comment into six toxicity categories using multi-label classification.
Each category must be assigned a binary value: 
• 1 = label is present
• 0 = label is not present

IMPORTANT:
•	Labels are NOT mutually exclusive.
•	A comment can have multiple labels = 1.
•	If the comment contains no toxic content, all labels must be 0.
•	Do NOT provide explanations.
•	Do NOT infer beyond the text.
•	Be conservative and evidence-based, especially for rare labels.

LABEL DEFINITIONS:
1.	toxic: General rude, disrespectful, or offensive language.
2.	severe_toxic: Extremely aggressive, abusive, or highly hostile language.
3.	obscene: Profanity, vulgar, or sexually explicit language.
4.	threat: Statements that imply violence, harm, or intimidation.
5.	insult: Direct personal attacks, humiliation, or degrading remarks toward a person or group.
6.	identity_hate: Hate speech targeting protected characteristics (e.g., race, religion, gender, nationality, ethnicity, sexual orientation).

DECISION GUIDELINES (CRITICAL FOR CONSISTENCY):
•	Assign "threat" = 1 only if there is a clear indication of harm or violence.
•	Assign "identity_hate" = 1 only if hatred targets an identity group, not just an individual.
•	Profanity alone → usually "obscene" and/or "toxic", not automatically "severe_toxic".
•	Strong personal attack → "insult" (and possibly "toxic").
•	Extremely abusive language → may include "severe_toxic" in addition to other labels.
•	Absence of offensive cues → all zeros.

OUTPUT FORMAT (STRICT — JSON ONLY):
Return ONLY a valid JSON object with EXACTLY these keys:
{{
"toxic": 0 or 1,
"severe_toxic": 0 or 1,
"obscene": 0 or 1,
"threat": 0 or 1,
"insult": 0 or 1,
"identity_hate": 0 or 1
}}
NO extra text. NO explanations. NO additional fields.

Comment: "{comment_text}" """

# ─────────────────────────────────────────────
# FIX 1: COMMENT SANITIZER
# ─────────────────────────────────────────────
def sanitize_comment(text: str) -> str:
    """
    Prepare comment text for safe injection into the prompt string.
    - Replaces newlines with a space so the comment stays on one logical line
    - Escapes { and } so Python's .format() doesn't treat them as placeholders
    - Strips leading/trailing whitespace
    """
    text = text.replace("\n", " ").replace("\r", " ")   # flatten multi-line
    text = text.replace("{", "(").replace("}", ")")      # escape braces
    return text.strip()


# ─────────────────────────────────────────────
# FIX 2: ROBUST JSON EXTRACTOR
# ─────────────────────────────────────────────
def extract_json(raw: str) -> dict:
    """
    Extract the first JSON object found in raw string.
    Handles cases where GPT prepends/appends explanation text or markdown fences.
    Raises json.JSONDecodeError if no valid JSON object is found.
    """
    # Strip markdown fences just in case (```json ... ```)
    raw = re.sub(r"```(?:json)?", "", raw).strip()

    # Find the first {...} block (non-nested search is fine for flat label dicts)
    match = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if not match:
        raise json.JSONDecodeError("No JSON object found in response", raw, 0)

    return json.loads(match.group())


# ─────────────────────────────────────────────
# CHECKPOINT HELPERS
# ─────────────────────────────────────────────
def get_checkpoint_path(batch_idx: int) -> str:
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    return f"{CHECKPOINT_DIR}/batch_{batch_idx:06d}.json"

def save_checkpoint(batch_idx: int, predictions: list):
    path = get_checkpoint_path(batch_idx)
    with open(path, "w") as f:
        json.dump(predictions, f)

def load_all_checkpoints() -> dict:
    """Load all existing checkpoint files. Returns {batch_idx: predictions}."""
    checkpoints = {}
    checkpoint_dir = Path(CHECKPOINT_DIR)
    if not checkpoint_dir.exists():
        return checkpoints
    for file in sorted(checkpoint_dir.glob("batch_*.json")):
        batch_idx = int(file.stem.split("_")[1])
        with open(file) as f:
            checkpoints[batch_idx] = json.load(f)
    return checkpoints

def clear_checkpoints():
    checkpoint_dir = Path(CHECKPOINT_DIR)
    if checkpoint_dir.exists():
        for file in checkpoint_dir.glob("batch_*.json"):
            file.unlink()


# ─────────────────────────────────────────────
# ASYNC API CALL  (all 4 fixes applied here)
# ─────────────────────────────────────────────
async def classify_comment_async(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    comment_text: str,
    index: int,
) -> tuple:
    """Classify a single comment asynchronously with retry logic."""

    # ── FIX 1: sanitize before injecting into prompt ──────────────────────────
    clean_text = sanitize_comment(comment_text)
    prompt = ZERO_SHOT_PROMPT.format(comment_text=clean_text)

    payload = {
        "model": MODEL,
        "messages": [
            # ── FIX 4: system message enforces JSON-only output ───────────────
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user",   "content": prompt},
        ],
        "temperature": 0,
        "max_tokens": 150,   # ── FIX 3: raised from 100 → 150 (prevents truncation)
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    async with semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                async with session.post(
                    "https://api.openai.com/v1/chat/completions",
                    json=payload,
                    headers=headers,
                    timeout=aiohttp.ClientTimeout(total=30),
                ) as resp:

                    if resp.status == 429:
                        wait = RETRY_DELAY * (2 ** attempt)
                        print(f"  [RateLimit] idx={index} | waiting {wait}s...")
                        await asyncio.sleep(wait)
                        continue

                    data = await resp.json()
                    raw  = data["choices"][0]["message"]["content"].strip()

                    # ── FIX 2: robust JSON extraction ─────────────────────────
                    result = extract_json(raw)
                    return index, {label: int(result.get(label, 0)) for label in LABELS}

            except json.JSONDecodeError as e:
                # Log the raw response so you can debug what GPT actually returned
                raw_preview = locals().get("raw", "<no response captured>")[:200]
                print(
                    f"  [Warning] JSON parse error at idx={index}, attempt {attempt + 1} | "
                    f"raw preview: {repr(raw_preview)}"
                )
            except Exception as e:
                print(f"  [Error] idx={index}, attempt {attempt + 1}: {e}")

            await asyncio.sleep(RETRY_DELAY * (attempt + 1))

    print(f"  [Fallback] All-zero prediction for idx={index}")
    return index, {label: 0 for label in LABELS}


# ─────────────────────────────────────────────
# ASYNC BATCH PROCESSOR
# ─────────────────────────────────────────────
async def classify_batch_async(
    comments: list,
    batch_size: int = BATCH_SIZE,
    max_concurrent: int = MAX_CONCURRENT,
) -> list:
    """
    Classify all comments using async parallel API calls.
    - Checkpoints every batch_size rows (crash-safe)
    - Resumes automatically from last checkpoint
    - Runs up to max_concurrent calls simultaneously
    """
    total      = len(comments)
    num_batches = (total + batch_size - 1) // batch_size

    print(f"\n{'='*62}")
    print(f"  Async Toxicity Classifier — GPT-4.1")
    print(f"  Total Comments  : {total:,}")
    print(f"  Batch Size      : {batch_size} (checkpoint every N rows)")
    print(f"  Total Batches   : {num_batches}")
    print(f"  Concurrency     : {max_concurrent} parallel API calls")
    print(f"  Est. Runtime    : ~{total / (max_concurrent * 3) / 60:.0f}–{total / (max_concurrent * 2) / 60:.0f} min")
    print(f"  Checkpoint Dir  : {CHECKPOINT_DIR}")
    print(f"{'='*62}\n")

    # Load existing checkpoints
    existing = load_all_checkpoints()
    if existing:
        print(f"  ✅ Resuming: {len(existing)}/{num_batches} batches already done.\n")

    # Pre-allocate results
    all_results = [None] * total
    for batch_idx, preds in existing.items():
        start = batch_idx * batch_size
        for i, pred in enumerate(preds):
            if start + i < total:
                all_results[start + i] = pred

    semaphore = asyncio.Semaphore(max_concurrent)
    connector = aiohttp.TCPConnector(limit=max_concurrent)

    async with aiohttp.ClientSession(connector=connector) as session:
        for batch_idx in range(num_batches):
            if batch_idx in existing:
                start = batch_idx * batch_size
                end   = min(start + batch_size, total)
                print(f"[Batch {batch_idx+1:>4}/{num_batches}] ⏭ Skipped (checkpoint exists) | rows {start+1:,}–{end:,}")
                continue

            start          = batch_idx * batch_size
            end            = min(start + batch_size, total)
            batch_comments = comments[start:end]

            print(f"[Batch {batch_idx+1:>4}/{num_batches}] 🔄 rows {start+1:,}–{end:,} ({len(batch_comments)} comments)...")
            batch_start = time.time()

            tasks = [
                classify_comment_async(session, semaphore, comment, start + i)
                for i, comment in enumerate(batch_comments)
            ]
            batch_results_raw = await asyncio.gather(*tasks)
            batch_results_raw = sorted(batch_results_raw, key=lambda x: x[0])
            batch_preds       = [pred for _, pred in batch_results_raw]

            for i, pred in enumerate(batch_preds):
                all_results[start + i] = pred

            elapsed          = time.time() - batch_start
            rate             = len(batch_comments) / elapsed
            remaining_batches = num_batches - (batch_idx + 1)
            eta_min          = (remaining_batches * elapsed) / 60

            print(f"  ✓ Done | {elapsed:.1f}s | {rate:.1f} comments/s | ETA: ~{eta_min:.1f} min remaining\n")
            save_checkpoint(batch_idx, batch_preds)

    # Safety check
    missing = sum(1 for r in all_results if r is None)
    if missing:
        print(f"  [Warning] {missing} comments missing predictions. Filling with zeros.")
        all_results = [r if r is not None else {label: 0 for label in LABELS} for r in all_results]

    print(f"\n✅ Classification complete. Total: {len(all_results):,} predictions.\n")
    return all_results


# ─────────────────────────────────────────────
# EVALUATION METRICS
# ─────────────────────────────────────────────
def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_names: list = LABELS,
) -> dict:
    """Compute full multi-label evaluation metrics."""
    print(f"\n{'='*62}")
    print("  EVALUATION METRICS (Multi-Label)")
    print(f"{'='*62}")

    exact_match_acc    = accuracy_score(y_true, y_pred)
    precision_micro    = precision_score(y_true, y_pred, average="micro",     zero_division=0)
    recall_micro       = recall_score(y_true, y_pred, average="micro",        zero_division=0)
    f1_micro           = f1_score(y_true, y_pred, average="micro",            zero_division=0)
    precision_macro    = precision_score(y_true, y_pred, average="macro",     zero_division=0)
    recall_macro       = recall_score(y_true, y_pred, average="macro",        zero_division=0)
    f1_macro           = f1_score(y_true, y_pred, average="macro",            zero_division=0)
    precision_weighted = precision_score(y_true, y_pred, average="weighted",  zero_division=0)
    recall_weighted    = recall_score(y_true, y_pred, average="weighted",     zero_division=0)
    f1_weighted        = f1_score(y_true, y_pred, average="weighted",         zero_division=0)
    per_label_precision = precision_score(y_true, y_pred, average=None,       zero_division=0)
    per_label_recall    = recall_score(y_true, y_pred, average=None,          zero_division=0)
    per_label_f1        = f1_score(y_true, y_pred, average=None,              zero_division=0)

    print(f"\n  Exact Match Accuracy (Subset) : {exact_match_acc:.4f}")
    print(f"\n  {'Metric':<12} {'Micro':>8} {'Macro':>8} {'Weighted':>10}")
    print(f"  {'-'*42}")
    print(f"  {'Precision':<12} {precision_micro:>8.4f} {precision_macro:>8.4f} {precision_weighted:>10.4f}")
    print(f"  {'Recall':<12} {recall_micro:>8.4f} {recall_macro:>8.4f} {recall_weighted:>10.4f}")
    print(f"  {'F1 Score':<12} {f1_micro:>8.4f} {f1_macro:>8.4f} {f1_weighted:>10.4f}")
    print(f"\n  Per-Label Breakdown:")
    print(f"  {'Label':<15} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
    print(f"  {'-'*55}")
    for i, label in enumerate(label_names):
        support = int(y_true[:, i].sum())
        print(f"  {label:<15} {per_label_precision[i]:>10.4f} {per_label_recall[i]:>8.4f} {per_label_f1[i]:>8.4f} {support:>9}")
    print(f"\n{'='*62}\n")

    return {
        "exact_match_accuracy": exact_match_acc,
        "precision_micro":      precision_micro,
        "recall_micro":         recall_micro,
        "f1_micro":             f1_micro,
        "precision_macro":      precision_macro,
        "recall_macro":         recall_macro,
        "f1_macro":             f1_macro,
        "precision_weighted":   precision_weighted,
        "recall_weighted":      recall_weighted,
        "f1_weighted":          f1_weighted,
        "per_label": {
            label: {
                "precision": float(per_label_precision[i]),
                "recall":    float(per_label_recall[i]),
                "f1":        float(per_label_f1[i]),
                "support":   int(y_true[:, i].sum()),
            }
            for i, label in enumerate(label_names)
        },
    }


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────
async def run_pipeline(
    input_path:  str,
    comment_col: str = "comment_text",
    label_cols:  Optional[list] = None,
    output_path: str = OUTPUT_PATH,
    evaluate:    bool = True,
):
    total_start = time.time()

    # 1. Load data
    print(f"\nLoading data from:\n  {input_path}")
    df = pd.read_csv(input_path)
    print(f"Loaded {len(df):,} rows.\n")
    comments = df[comment_col].fillna("").tolist()

    # 2. Run async classifier
    predictions = await classify_batch_async(
        comments,
        batch_size=BATCH_SIZE,
        max_concurrent=MAX_CONCURRENT,
    )

    # 3. Build result dataframe
    pred_df = pd.DataFrame(predictions)
    pred_df.columns = [f"pred_{l}" for l in LABELS]
    result_df = pd.concat([df.reset_index(drop=True), pred_df], axis=1)

    # 4. Save predictions
    result_df.to_csv(output_path, index=False)
    print(f"📄 Predictions saved:\n  {output_path}")

    # 5. Evaluate
    metrics = None
    if evaluate and label_cols:
        missing_cols = [c for c in label_cols if c not in df.columns]
        if missing_cols:
            print(f"[Warning] Ground-truth columns not found: {missing_cols}. Skipping evaluation.")
        else:
            y_true  = df[label_cols].values.astype(int)
            y_pred  = pred_df.values.astype(int)
            metrics = evaluate_predictions(y_true, y_pred, label_names=label_cols)
            metrics_path = output_path.replace(".csv", "_metrics.json")
            with open(metrics_path, "w") as f:
                json.dump(metrics, f, indent=2)
            print(f"📊 Metrics saved:\n  {metrics_path}")

    # 6. Clean up checkpoints
    print("\n🧹 Cleaning up checkpoint files...")
    clear_checkpoints()

    total_elapsed = time.time() - total_start
    print(f"\n⏱ Total runtime : {total_elapsed / 60:.1f} minutes")
    print("🎉 Pipeline complete!\n")
    return result_df, metrics


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    asyncio.run(
        run_pipeline(
            input_path=INPUT_PATH,
            comment_col="comment_text",   # ← update if column name differs
            label_cols=LABELS,            # ← set to None if no ground-truth labels
            output_path=OUTPUT_PATH,
            evaluate=True,
        )
    )


Loading data from:
  /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/stress_test_eval_set.csv
Loaded 8,000 rows.


  Async Toxicity Classifier — GPT-4.1
  Total Comments  : 8,000
  Batch Size      : 500 (checkpoint every N rows)
  Total Batches   : 16
  Concurrency     : 20 parallel API calls
  Est. Runtime    : ~2–3 min
  Checkpoint Dir  : /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/checkpoints

[Batch    1/16] 🔄 rows 1–500 (500 comments)...
  ✓ Done | 31.9s | 15.7 comments/s | ETA: ~8.0 min remaining

[Batch    2/16] 🔄 rows 501–1,000 (500 comments)...
  ✓ Done | 32.8s | 15.3 comments/s | ETA: ~7.6 min remaining

[Batch    3/16] 🔄 rows 1,001–1,500 (500 comments)...
  ✓ Done | 33.8s | 14.8 comments/s | ETA: ~7.3 min remaining

[Batch    4/16] 🔄 rows 1,501–2,000 (500 comments)...
  ✓ Done | 34.0s | 14.7 comments/s | ETA: ~6.8 min remaining

[Batch    5/16] 🔄 rows 2,001–2,500 (500 comments)...
  ✓ Done | 33.5s

# 8000 STRATIFIED

In [ ]:
"""
Toxicity Multi-Label Classifier using GPT-4.1 API
Business Analytics Tool | Zero-Shot Prompt
Features: Async Parallel Calls (20x faster) + Checkpoint/Resume (crash-safe)

FIXES APPLIED:
  1. Comment sanitization — strips newlines, escapes curly braces before .format()
  2. Robust JSON extraction — regex pulls JSON even if GPT adds extra text
  3. max_tokens increased 100 → 150 to prevent truncated JSON responses
  4. System message added to reinforce JSON-only output from the model
  5. Retry logic now re-raises cleanly and logs the raw response on parse failure
"""

import os
import re
import json
import time
import asyncio
import aiohttp
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from typing import Optional

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = "gpt-4.1"
BATCH_SIZE = 500          # Checkpoint saved every 500 rows
MAX_CONCURRENT = 20       # Parallel async API calls
MAX_RETRIES = 3
RETRY_DELAY = 2           # seconds (doubles on each retry)
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Paths — update BASE_DIR to your local folder
BASE_DIR = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment"
INPUT_PATH  = f"{BASE_DIR}/few_shot_pool_stratifiedcode.csv"
OUTPUT_PATH = f"{BASE_DIR}/predictions_output_stratified.csv"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"

# ── FIX 4: system message reinforces JSON-only output ──────────────────────────
SYSTEM_MESSAGE = (
    "You are a JSON-only classifier. "
    "Respond ONLY with a valid JSON object. "
    "No explanation, no markdown fences, no extra text before or after the JSON."
)

ZERO_SHOT_PROMPT = """SYSTEM ROLE: You are a strict multi-label text classifier specialising in toxic content detection for online comments.

TASK: Classify the given comment into six toxicity categories using multi-label classification.
Each category must be assigned a binary value: 
• 1 = label is present
• 0 = label is not present

IMPORTANT:
•	Labels are NOT mutually exclusive.
•	A comment can have multiple labels = 1.
•	If the comment contains no toxic content, all labels must be 0.
•	Do NOT provide explanations.
•	Do NOT infer beyond the text.
•	Be conservative and evidence-based, especially for rare labels.

LABEL DEFINITIONS:
1.	toxic: General rude, disrespectful, or offensive language.
2.	severe_toxic: Extremely aggressive, abusive, or highly hostile language.
3.	obscene: Profanity, vulgar, or sexually explicit language.
4.	threat: Statements that imply violence, harm, or intimidation.
5.	insult: Direct personal attacks, humiliation, or degrading remarks toward a person or group.
6.	identity_hate: Hate speech targeting protected characteristics (e.g., race, religion, gender, nationality, ethnicity, sexual orientation).

DECISION GUIDELINES (CRITICAL FOR CONSISTENCY):
•	Assign "threat" = 1 only if there is a clear indication of harm or violence.
•	Assign "identity_hate" = 1 only if hatred targets an identity group, not just an individual.
•	Profanity alone → usually "obscene" and/or "toxic", not automatically "severe_toxic".
•	Strong personal attack → "insult" (and possibly "toxic").
•	Extremely abusive language → may include "severe_toxic" in addition to other labels.
•	Absence of offensive cues → all zeros.

OUTPUT FORMAT (STRICT — JSON ONLY):
Return ONLY a valid JSON object with EXACTLY these keys:
{{
"toxic": 0 or 1,
"severe_toxic": 0 or 1,
"obscene": 0 or 1,
"threat": 0 or 1,
"insult": 0 or 1,
"identity_hate": 0 or 1
}}
NO extra text. NO explanations. NO additional fields.

Comment: "{comment_text}" """

# ─────────────────────────────────────────────
# FIX 1: COMMENT SANITIZER
# ─────────────────────────────────────────────
def sanitize_comment(text: str) -> str:
    """
    Prepare comment text for safe injection into the prompt string.
    - Replaces newlines with a space so the comment stays on one logical line
    - Escapes { and } so Python's .format() doesn't treat them as placeholders
    - Strips leading/trailing whitespace
    """
    text = text.replace("\n", " ").replace("\r", " ")   # flatten multi-line
    text = text.replace("{", "(").replace("}", ")")      # escape braces
    return text.strip()


# ─────────────────────────────────────────────
# FIX 2: ROBUST JSON EXTRACTOR
# ─────────────────────────────────────────────
def extract_json(raw: str) -> dict:
    """
    Extract the first JSON object found in raw string.
    Handles cases where GPT prepends/appends explanation text or markdown fences.
    Raises json.JSONDecodeError if no valid JSON object is found.
    """
    # Strip markdown fences just in case (```json ... ```)
    raw = re.sub(r"```(?:json)?", "", raw).strip()

    # Find the first {...} block (non-nested search is fine for flat label dicts)
    match = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if not match:
        raise json.JSONDecodeError("No JSON object found in response", raw, 0)

    return json.loads(match.group())


# ─────────────────────────────────────────────
# CHECKPOINT HELPERS
# ─────────────────────────────────────────────
def get_checkpoint_path(batch_idx: int) -> str:
    Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    return f"{CHECKPOINT_DIR}/batch_{batch_idx:06d}.json"

def save_checkpoint(batch_idx: int, predictions: list):
    path = get_checkpoint_path(batch_idx)
    with open(path, "w") as f:
        json.dump(predictions, f)

def load_all_checkpoints() -> dict:
    """Load all existing checkpoint files. Returns {batch_idx: predictions}."""
    checkpoints = {}
    checkpoint_dir = Path(CHECKPOINT_DIR)
    if not checkpoint_dir.exists():
        return checkpoints
    for file in sorted(checkpoint_dir.glob("batch_*.json")):
        batch_idx = int(file.stem.split("_")[1])
        with open(file) as f:
            checkpoints[batch_idx] = json.load(f)
    return checkpoints

def clear_checkpoints():
    checkpoint_dir = Path(CHECKPOINT_DIR)
    if checkpoint_dir.exists():
        for file in checkpoint_dir.glob("batch_*.json"):
            file.unlink()


# ─────────────────────────────────────────────
# ASYNC API CALL  (all 4 fixes applied here)
# ─────────────────────────────────────────────
async def classify_comment_async(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    comment_text: str,
    index: int,
) -> tuple:
    """Classify a single comment asynchronously with retry logic."""

    # ── FIX 1: sanitize before injecting into prompt ──────────────────────────
    clean_text = sanitize_comment(comment_text)
    prompt = ZERO_SHOT_PROMPT.format(comment_text=clean_text)

    payload = {
        "model": MODEL,
        "messages": [
            # ── FIX 4: system message enforces JSON-only output ───────────────
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user",   "content": prompt},
        ],
        "temperature": 0,
        "max_tokens": 150,   # ── FIX 3: raised from 100 → 150 (prevents truncation)
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    async with semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                async with session.post(
                    "https://api.openai.com/v1/chat/completions",
                    json=payload,
                    headers=headers,
                    timeout=aiohttp.ClientTimeout(total=30),
                ) as resp:

                    if resp.status == 429:
                        wait = RETRY_DELAY * (2 ** attempt)
                        print(f"  [RateLimit] idx={index} | waiting {wait}s...")
                        await asyncio.sleep(wait)
                        continue

                    data = await resp.json()
                    raw  = data["choices"][0]["message"]["content"].strip()

                    # ── FIX 2: robust JSON extraction ─────────────────────────
                    result = extract_json(raw)
                    return index, {label: int(result.get(label, 0)) for label in LABELS}

            except json.JSONDecodeError as e:
                # Log the raw response so you can debug what GPT actually returned
                raw_preview = locals().get("raw", "<no response captured>")[:200]
                print(
                    f"  [Warning] JSON parse error at idx={index}, attempt {attempt + 1} | "
                    f"raw preview: {repr(raw_preview)}"
                )
            except Exception as e:
                print(f"  [Error] idx={index}, attempt {attempt + 1}: {e}")

            await asyncio.sleep(RETRY_DELAY * (attempt + 1))

    print(f"  [Fallback] All-zero prediction for idx={index}")
    return index, {label: 0 for label in LABELS}


# ─────────────────────────────────────────────
# ASYNC BATCH PROCESSOR
# ─────────────────────────────────────────────
async def classify_batch_async(
    comments: list,
    batch_size: int = BATCH_SIZE,
    max_concurrent: int = MAX_CONCURRENT,
) -> list:
    """
    Classify all comments using async parallel API calls.
    - Checkpoints every batch_size rows (crash-safe)
    - Resumes automatically from last checkpoint
    - Runs up to max_concurrent calls simultaneously
    """
    total      = len(comments)
    num_batches = (total + batch_size - 1) // batch_size

    print(f"\n{'='*62}")
    print(f"  Async Toxicity Classifier — GPT-4.1")
    print(f"  Total Comments  : {total:,}")
    print(f"  Batch Size      : {batch_size} (checkpoint every N rows)")
    print(f"  Total Batches   : {num_batches}")
    print(f"  Concurrency     : {max_concurrent} parallel API calls")
    print(f"  Est. Runtime    : ~{total / (max_concurrent * 3) / 60:.0f}–{total / (max_concurrent * 2) / 60:.0f} min")
    print(f"  Checkpoint Dir  : {CHECKPOINT_DIR}")
    print(f"{'='*62}\n")

    # Load existing checkpoints
    existing = load_all_checkpoints()
    if existing:
        print(f"  ✅ Resuming: {len(existing)}/{num_batches} batches already done.\n")

    # Pre-allocate results
    all_results = [None] * total
    for batch_idx, preds in existing.items():
        start = batch_idx * batch_size
        for i, pred in enumerate(preds):
            if start + i < total:
                all_results[start + i] = pred

    semaphore = asyncio.Semaphore(max_concurrent)
    connector = aiohttp.TCPConnector(limit=max_concurrent)

    async with aiohttp.ClientSession(connector=connector) as session:
        for batch_idx in range(num_batches):
            if batch_idx in existing:
                start = batch_idx * batch_size
                end   = min(start + batch_size, total)
                print(f"[Batch {batch_idx+1:>4}/{num_batches}] ⏭ Skipped (checkpoint exists) | rows {start+1:,}–{end:,}")
                continue

            start          = batch_idx * batch_size
            end            = min(start + batch_size, total)
            batch_comments = comments[start:end]

            print(f"[Batch {batch_idx+1:>4}/{num_batches}] 🔄 rows {start+1:,}–{end:,} ({len(batch_comments)} comments)...")
            batch_start = time.time()

            tasks = [
                classify_comment_async(session, semaphore, comment, start + i)
                for i, comment in enumerate(batch_comments)
            ]
            batch_results_raw = await asyncio.gather(*tasks)
            batch_results_raw = sorted(batch_results_raw, key=lambda x: x[0])
            batch_preds       = [pred for _, pred in batch_results_raw]

            for i, pred in enumerate(batch_preds):
                all_results[start + i] = pred

            elapsed          = time.time() - batch_start
            rate             = len(batch_comments) / elapsed
            remaining_batches = num_batches - (batch_idx + 1)
            eta_min          = (remaining_batches * elapsed) / 60

            print(f"  ✓ Done | {elapsed:.1f}s | {rate:.1f} comments/s | ETA: ~{eta_min:.1f} min remaining\n")
            save_checkpoint(batch_idx, batch_preds)

    # Safety check
    missing = sum(1 for r in all_results if r is None)
    if missing:
        print(f"  [Warning] {missing} comments missing predictions. Filling with zeros.")
        all_results = [r if r is not None else {label: 0 for label in LABELS} for r in all_results]

    print(f"\n✅ Classification complete. Total: {len(all_results):,} predictions.\n")
    return all_results


# ─────────────────────────────────────────────
# EVALUATION METRICS
# ─────────────────────────────────────────────
def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_names: list = LABELS,
) -> dict:
    """Compute full multi-label evaluation metrics."""
    print(f"\n{'='*62}")
    print("  EVALUATION METRICS (Multi-Label)")
    print(f"{'='*62}")

    exact_match_acc    = accuracy_score(y_true, y_pred)
    precision_micro    = precision_score(y_true, y_pred, average="micro",     zero_division=0)
    recall_micro       = recall_score(y_true, y_pred, average="micro",        zero_division=0)
    f1_micro           = f1_score(y_true, y_pred, average="micro",            zero_division=0)
    precision_macro    = precision_score(y_true, y_pred, average="macro",     zero_division=0)
    recall_macro       = recall_score(y_true, y_pred, average="macro",        zero_division=0)
    f1_macro           = f1_score(y_true, y_pred, average="macro",            zero_division=0)
    precision_weighted = precision_score(y_true, y_pred, average="weighted",  zero_division=0)
    recall_weighted    = recall_score(y_true, y_pred, average="weighted",     zero_division=0)
    f1_weighted        = f1_score(y_true, y_pred, average="weighted",         zero_division=0)
    per_label_precision = precision_score(y_true, y_pred, average=None,       zero_division=0)
    per_label_recall    = recall_score(y_true, y_pred, average=None,          zero_division=0)
    per_label_f1        = f1_score(y_true, y_pred, average=None,              zero_division=0)

    print(f"\n  Exact Match Accuracy (Subset) : {exact_match_acc:.4f}")
    print(f"\n  {'Metric':<12} {'Micro':>8} {'Macro':>8} {'Weighted':>10}")
    print(f"  {'-'*42}")
    print(f"  {'Precision':<12} {precision_micro:>8.4f} {precision_macro:>8.4f} {precision_weighted:>10.4f}")
    print(f"  {'Recall':<12} {recall_micro:>8.4f} {recall_macro:>8.4f} {recall_weighted:>10.4f}")
    print(f"  {'F1 Score':<12} {f1_micro:>8.4f} {f1_macro:>8.4f} {f1_weighted:>10.4f}")
    print(f"\n  Per-Label Breakdown:")
    print(f"  {'Label':<15} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
    print(f"  {'-'*55}")
    for i, label in enumerate(label_names):
        support = int(y_true[:, i].sum())
        print(f"  {label:<15} {per_label_precision[i]:>10.4f} {per_label_recall[i]:>8.4f} {per_label_f1[i]:>8.4f} {support:>9}")
    print(f"\n{'='*62}\n")

    return {
        "exact_match_accuracy": exact_match_acc,
        "precision_micro":      precision_micro,
        "recall_micro":         recall_micro,
        "f1_micro":             f1_micro,
        "precision_macro":      precision_macro,
        "recall_macro":         recall_macro,
        "f1_macro":             f1_macro,
        "precision_weighted":   precision_weighted,
        "recall_weighted":      recall_weighted,
        "f1_weighted":          f1_weighted,
        "per_label": {
            label: {
                "precision": float(per_label_precision[i]),
                "recall":    float(per_label_recall[i]),
                "f1":        float(per_label_f1[i]),
                "support":   int(y_true[:, i].sum()),
            }
            for i, label in enumerate(label_names)
        },
    }


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────
async def run_pipeline(
    input_path:  str,
    comment_col: str = "comment_text",
    label_cols:  Optional[list] = None,
    output_path: str = OUTPUT_PATH,
    evaluate:    bool = True,
):
    total_start = time.time()

    # 1. Load data
    print(f"\nLoading data from:\n  {input_path}")
    df = pd.read_csv(input_path)
    print(f"Loaded {len(df):,} rows.\n")
    comments = df[comment_col].fillna("").tolist()

    # 2. Run async classifier
    predictions = await classify_batch_async(
        comments,
        batch_size=BATCH_SIZE,
        max_concurrent=MAX_CONCURRENT,
    )

    # 3. Build result dataframe
    pred_df = pd.DataFrame(predictions)
    pred_df.columns = [f"pred_{l}" for l in LABELS]
    result_df = pd.concat([df.reset_index(drop=True), pred_df], axis=1)

    # 4. Save predictions
    result_df.to_csv(output_path, index=False)
    print(f"📄 Predictions saved:\n  {output_path}")

    # 5. Evaluate
    metrics = None
    if evaluate and label_cols:
        missing_cols = [c for c in label_cols if c not in df.columns]
        if missing_cols:
            print(f"[Warning] Ground-truth columns not found: {missing_cols}. Skipping evaluation.")
        else:
            y_true  = df[label_cols].values.astype(int)
            y_pred  = pred_df.values.astype(int)
            metrics = evaluate_predictions(y_true, y_pred, label_names=label_cols)
            metrics_path = output_path.replace(".csv", "_metrics.json")
            with open(metrics_path, "w") as f:
                json.dump(metrics, f, indent=2)
            print(f"📊 Metrics saved:\n  {metrics_path}")

    # 6. Clean up checkpoints
    print("\n🧹 Cleaning up checkpoint files...")
    clear_checkpoints()

    total_elapsed = time.time() - total_start
    print(f"\n⏱ Total runtime : {total_elapsed / 60:.1f} minutes")
    print("🎉 Pipeline complete!\n")
    return result_df, metrics


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    asyncio.run(
        run_pipeline(
            input_path=INPUT_PATH,
            comment_col="comment_text",   # ← update if column name differs
            label_cols=LABELS,            # ← set to None if no ground-truth labels
            output_path=OUTPUT_PATH,
            evaluate=True,
        )
    )


Loading data from:
  /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/few_shot_pool_stratifiedcode.csv
Loaded 8,002 rows.


  Async Toxicity Classifier — GPT-4.1
  Total Comments  : 8,002
  Batch Size      : 500 (checkpoint every N rows)
  Total Batches   : 17
  Concurrency     : 20 parallel API calls
  Est. Runtime    : ~2–3 min
  Checkpoint Dir  : /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/checkpoints

[Batch    1/17] 🔄 rows 1–500 (500 comments)...
  ✓ Done | 26.3s | 19.0 comments/s | ETA: ~7.0 min remaining

[Batch    2/17] 🔄 rows 501–1,000 (500 comments)...
  ✓ Done | 30.9s | 16.2 comments/s | ETA: ~7.7 min remaining

[Batch    3/17] 🔄 rows 1,001–1,500 (500 comments)...
  ✓ Done | 29.3s | 17.1 comments/s | ETA: ~6.8 min remaining

[Batch    4/17] 🔄 rows 1,501–2,000 (500 comments)...
  ✓ Done | 29.0s | 17.2 comments/s | ETA: ~6.3 min remaining

[Batch    5/17] 🔄 rows 2,001–2,500 (500 comments)...
  ✓ Done